## Set up

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sqlite3

con = sqlite3.connect("chicago_analysis.db")
%load_ext sql
%sql sqlite:///chicago_analysis.db

#### -Top 10 community areas by crime count

In [ ]:
sql_statement = """
    WITH area_names AS (
        SELECT DISTINCT community_area_number, community_area_name
        FROM SCHOOLS_DATA
    )

    SELECT 
        a.community_area_name AS "Community Area",
        COUNT(c.id) AS "Number of Crimes"
    FROM area_names a
    JOIN CRIME_DATA c
        ON a.community_area_number = CAST(c.community_area AS INTEGER)
    WHERE c.community_area IS NOT NULL
    GROUP BY a.community_area_number
    ORDER BY COUNT(c.id) DESC
    LIMIT 10
"""

df = pd.read_sql_query(sql_statement, con)


sns.barplot(
    data=df,
    x="Community Area",
    y="Number of Crimes",
)
plt.title('Top 10 community areas by crime count')
plt.xlabel('Community Area')
plt.ylabel('Number of Crimes')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

#### -Hardship index vs Crime count

In [ ]:
sql_statement = ''' 
    WITH crimes_stats AS (
        SELECT 
            COUNT(*) AS crimes_count,
            community_area
        FROM CRIME_DATA
        WHERE community_area IS NOT NULL
        GROUP BY community_area        
    )
    SELECT 
        cd.community_area_name,
        cd.hardship_index,
        cs.crimes_count
    FROM CENSUS_DATA cd
    JOIN crimes_stats cs
    ON cd.ca = CAST(cs.community_area AS REAL)
    WHERE cd.community_area_name IS NOT NULL
    AND cd.hardship_index IS NOT NULL  
'''

df = pd.read_sql_query(sql_statement, con)

plt.figure(figsize=(10, 6))

sns.regplot(
    data=df,
    x="hardship_index",
    y="crimes_count",
    scatter_kws={"s": 80, "alpha": 0.7, "color": "steelblue"},
    line_kws={"color": "red", "linewidth": 1.5}
)

plt.title("Hardship Index vs Crime Count (with Trend Line)", fontsize=14)
plt.xlabel("Hardship Index", fontsize=12)
plt.ylabel("Crime Count", fontsize=12)
plt.tight_layout()
plt.show()

#### -Hardship index vs Avg school safety score

In [ ]:
sql_statement = ''' 
    WITH safety_stats AS (
        SELECT 
            ROUND(AVG(safety_score), 2) AS avg_safety_score,
            community_area_number
        FROM SCHOOLS_DATA
        WHERE safety_score IS NOT NULL
        AND community_area_number IS NOT NULL
        GROUP BY community_area_number
    )
    SELECT
        cd.hardship_index,
        ss.avg_safety_score
    FROM CENSUS_DATA cd
    JOIN safety_stats ss
    ON cd.ca = CAST(ss.community_area_number AS REAL)
    WHERE cd.hardship_index IS NOT NULL
'''

df = pd.read_sql_query(sql_statement, con)

plt.figure(figsize=(10, 6))

sns.regplot(
    data=df,
    x="avg_safety_score",
    y="hardship_index",
    scatter_kws={"s": 80, "alpha": 0.7, "color": "steelblue"},
    line_kws={"color": "red", "linewidth": 1.5}
)

plt.tight_layout()
plt.title("Hardship index vs Avg school safety score by Community Area", fontsize = 14)
plt.xlabel("Average Safety Score", fontsize = 12)
plt.ylabel("Hardship Index", fontsize = 12)
plt.show()

#### -Crime types distribution (top 10)

In [ ]:
sql_statement = ''' 
    SELECT 
        primary_type,
        COUNT(primary_type) AS crime_count
    FROM CRIME_DATA
    WHERE primary_type IS NOT NULL
    GROUP BY primary_type
    ORDER BY crime_count DESC
    LIMIT 10
'''

df = pd.read_sql_query(sql_statement, con)

plt.figure(figsize=(12, 7))

sns.barplot(
    data=df,
    x="primary_type",
    y="crime_count"
)

plt.title("Top 10 Crime Types Distribution")
plt.xlabel("Crime Type")
plt.ylabel("Crime Count")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

#### -Community areas by all 3 risk metrics (Top 10)

In [ ]:
sql_statement = ''' 
    WITH crimes_stats AS (
    SELECT 
        COUNT(*) crimes_count,
        community_area
    FROM CRIME_DATA
    WHERE community_area IS NOT NULL
    GROUP BY community_area
),
school_stats AS (
    SELECT 
        ROUND(AVG(safety_score), 2) AS avg_school_safety_score,
        community_area_number,
        community_area_name
    FROM SCHOOLS_DATA
    WHERE safety_score IS NOT NULL
    AND community_area_number IS NOT NULL 
    AND community_area_name IS NOT NULL
    GROUP BY community_area_number 
)
SELECT 
    ss.community_area_name,
    ss.avg_school_safety_score,
    cs.crimes_count,
    cd.hardship_index
FROM school_stats ss
JOIN crimes_stats cs 
ON ss.community_area_number = CAST(cs.community_area AS INT)
JOIN CENSUS_DATA cd
ON cs.community_area = cd.ca
ORDER BY crimes_count DESC, hardship_index DESC, avg_school_safety_score ASC
LIMIT 10
'''

df = pd.read_sql_query(sql_statement, con)


heatmap_data = df.set_index("community_area_name")[["crimes_count", "hardship_index", "avg_school_safety_score"]]

heatmap_normalized = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())

plt.figure(figsize=(10, 8))

sns.heatmap(
    heatmap_normalized,
    annot=heatmap_data,   
    fmt=".1f",            
    cmap="YlOrRd",        
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Risk Level (normalized)"}
)

plt.title("Top 10 High-Risk Community Areas by Risk Metrics", fontsize=14)
plt.xlabel("Risk Metric")
plt.ylabel("Community Area")
plt.xticks(ticks=[0.5, 1.5, 2.5], labels=["Crime Count", "Hardship Index", "Avg Safety Score"])
plt.tight_layout()
plt.show()